# 03 Drop drill set features
Set features from the Combine PASS_RUSH_DROP drill, then two checks:
1. **Left vs right.** Each player has one rep to each side, so the two reps are compared feature by feature (ICC). This is the drill's reliability check.
2. **Drill vs game.** Player-level correlation between the drill features and the same features in games.

Window is 1.5 s (Logan, 10/27: start with the original 1.5 s). 1.2 s is computed alongside for later. Game features are the same four as `01_ol_passset`, rebuilt here from the saved window so this notebook stands alone.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from shared.config import PROCESSED
from shared.drill_viewer import align_reps
from shared.reliability import icc21, split_half

CLEAN = PROCESSED / 'clean'
RAW_PQ = PROCESSED / 'parquet'
KEYS = ['game_id', 'play_id', 'nfl_id']
FEATS = ['retreat_depth', 'lateral_shift', 'peak_speed', 'peak_accel_05']

## Drill reps
OL players, PASS_RUSH_DROP only, each rep shifted to start at (0, 0). Time rounded to 0.1 s so frames line up.

In [ ]:
cr = pd.read_parquet(RAW_PQ / 'nfl-big-data-bowl-2027__combine_results.parquet',
                     columns=['nfl_id', 'combine_position'])
ol_ids = cr.loc[cr['combine_position'].eq('OL'), 'nfl_id']

ct = pd.read_parquet(CLEAN / 'combine_tracking.parquet')
drop = align_reps(ct, ol_ids)
drop = drop[drop['drill_name'].eq('PASS_RUSH_DROP')].copy()
drop['t'] = drop['t'].round(1)
drop['event_id'].nunique(), drop['nfl_id'].nunique()

Check: the clip starts with the player already moving (median start speed about 0.35 yd/s; most reach 0.5 yd/s by 0.1 s). There is no stance in the clip, so the drill window starts at clip start, while the game window starts at the snap after a stance. Keep in mind for first-step acceleration.

In [ ]:
drop.groupby('event_id').agg(start_speed=('s', 'first')).describe().round(2).T

## Features per rep
Same definitions as the game side, with the clip start standing in for the stance:
- `retreat_depth`: yards back at the end of the window (back is −x in this drill)
- `lateral_shift`: |yards sideways| at the end of the window
- `peak_speed`: max smoothed speed in the window
- `peak_accel_05`: max smoothed acceleration from 0 to 0.5 s
- `side`: which way he set, from y at 1.2 s (−1 or +1)
- `first_dip`: time of the first speed dip (first pad); `pad_in_window` flags reps where it lands inside the window

In [ ]:
def first_dip(s):
    s = s.to_numpy()
    for i in range(3, len(s) - 1):
        if s[i] < s[i - 1] and s[i] <= s[i + 1] and s[i] < s[:i].max() - 0.2:
            return i / 10
    return np.nan

def drill_features(drop, t_end):
    end = drop[drop['t'] == t_end].set_index('event_id')
    in_win = drop[drop['t'].between(0, t_end)].groupby('event_id')
    f = pd.DataFrame({
        'retreat_depth': -end['dx'],
        'lateral_shift': end['dy'].abs(),
        'peak_speed': in_win['s_sg'].max(),
        'peak_accel_05': drop[drop['t'].between(0, 0.5)].groupby('event_id')['a_sg'].max(),
    })
    info = drop.groupby('event_id').agg(nfl_id=('nfl_id', 'first'), rep=('rep', 'first'))
    info['side'] = np.sign(drop[drop['t'] == 1.2].set_index('event_id')['dy'])
    info['first_dip'] = drop.groupby('event_id')['s_sg'].apply(first_dip)
    f = f.join(info)
    f['pad_in_window'] = f['first_dip'] <= t_end
    return f

d15 = drill_features(drop, 1.5)
d12 = drill_features(drop, 1.2)
d15[FEATS].describe().round(2).T

In [ ]:
pd.Series({'reps': len(d15), 'pad inside 1.5 s window': d15['pad_in_window'].mean().round(3),
           'pad inside 1.2 s window': d12['pad_in_window'].mean().round(3)})

Side vs rep order. If rep 1 is almost always the same side, side and order (warm-up, learning) can't be separated.

In [ ]:
pd.crosstab(d15['rep'], d15['side'])

## Check 1: left vs right
Players with exactly one rep to each side. ICC(2,1) is absolute agreement, so a shift between sides lowers it; `r` ignores the shift. Sanity check first: the same ICC function on OL tracked-40 peak speed should be high (known retest ICC about 0.9).

In [ ]:
forty = align_reps(ct, ol_ids)
forty = forty[forty['drill_name'].eq('FORTY_YARD_DASH')]
f40 = forty.groupby(['nfl_id', 'rep'])['s_sg'].max().unstack().dropna()
round(icc21(f40[1], f40[2]), 2), len(f40)

In [ ]:
def left_right(f):
    both = f.groupby('nfl_id').filter(lambda g: len(g) == 2 and set(g['side']) == {-1, 1})
    left = both[both['side'] < 0].set_index('nfl_id')
    right = both[both['side'] > 0].set_index('nfl_id').loc[left.index]
    rows = {}
    for c in FEATS:
        rows[c] = {'ICC(2,1)': icc21(left[c], right[c]),
                   'r': np.corrcoef(left[c], right[c])[0, 1],
                   'mean left': left[c].mean(), 'mean right': right[c].mean()}
    return pd.DataFrame(rows).T.round(2), len(left)

left_right(d15)

In [ ]:
left_right(d12)

## Check 2: drill vs game
Game features rebuilt from the saved 1.5 s window, regular-season OL pass-pro plays with dropback over 2 s, players with 100+ plays. Game split-half reliability is shown first, overall and within position.

In [ ]:
win = pd.read_parquet(CLEAN / 'ol_passset_window.parquet')
plays = pd.read_parquet(CLEAN / 'player_play.parquet', columns=KEYS + ['dropback_duration'])
players = pd.read_parquet(RAW_PQ / 'nfl-big-data-bowl-2027__players.parquet', columns=['nfl_id', 'nfl_position'])

stance = win[win['t'] < 0].groupby(KEYS)[['x', 'y']].mean()
end = win[win['t'] == 1.5].set_index(KEYS)[['x', 'y']]
game = pd.DataFrame({
    'retreat_depth': stance['x'] - end['x'],
    'lateral_shift': (end['y'] - stance['y']).abs(),
    'peak_speed': win[win['t'].between(0, 1.5)].groupby(KEYS)['s_sg'].max(),
    'peak_accel_05': win[win['t'].between(0, 0.5)].groupby(KEYS)['a_sg'].max(),
}).reset_index().merge(plays, on=KEYS).merge(players, on='nfl_id')

game = game[game['dropback_duration'] > 2]
n_plays = game.groupby('nfl_id').size()
game = game[game['nfl_id'].isin(n_plays[n_plays >= 100].index)]
game[FEATS].describe().round(2).T

In [ ]:
pos = game.groupby('nfl_id')['nfl_position'].first()
within = game.copy()
for c in FEATS:
    within[c] = within[c] - within.groupby('nfl_position')[c].transform('mean')

pd.DataFrame({'overall': split_half(game, 'nfl_id', FEATS),
              'within position': split_half(within, 'nfl_id', FEATS)}).round(2)

In [ ]:
game_pl = game.groupby('nfl_id')[FEATS].mean()
drill_pl = d15.groupby('nfl_id')[FEATS].mean()          # average of the two sides
both = game_pl.join(drill_pl, lsuffix='_game', rsuffix='_drill', how='inner')
both['pos'] = pos

link = pd.Series({c: np.corrcoef(both[c + '_game'], both[c + '_drill'])[0, 1] for c in FEATS})
link_within = pd.Series({c: np.corrcoef(both[c + '_game'] - both.groupby('pos')[c + '_game'].transform('mean'),
                                        both[c + '_drill'] - both.groupby('pos')[c + '_drill'].transform('mean'))[0, 1]
                         for c in FEATS})
print('players with both:', len(both), '| rough 95% CI half-width for r near 0:', round(1.96 / np.sqrt(len(both) - 3), 2))
pd.DataFrame({'r': link, 'r within position': link_within}).round(2)

## Notes
- Interpretation and decisions go in `decisions.md`.
- A correlation can't be higher than the square root of the product of the two sides' reliabilities. With the drill's left-right ICC near zero, the drill-game link has almost no room to show up even if a real relationship exists.

In [ ]:
d15.to_parquet(CLEAN / 'drop_drill_features_1.5.parquet')
d12.to_parquet(CLEAN / 'drop_drill_features_1.2.parquet')